# 01 — Balance dashboard overview

**Pregunta humana:** ¿Cuál es la foto anual del sistema patrimonial?

Este reporte es ejecutivo. Integra operación, funding/distribuciones, posición operativa, deuda y QA, sin intentar reemplazar los reportes específicos.

Reglas:

- `Currency` queda como columna junto a la métrica/línea.
- ARS y USD conviven, pero nunca se suman.
- Caja real no se inventa: si no hay fuente frontend-safe, se muestra como `s/d`.
- Funding no es ingreso operativo.
- Deuda interna no es OPEX.
- Retiros/dividendos no son gastos de propiedad.


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 220)
pd.set_option("display.width", 260)

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "Makefile").exists() and (p / "accounting").is_dir():
        repo_root = p
        break
if repo_root is None:
    raise FileNotFoundError("Could not find repo root. Run from inside accounting-backend.")

reports_dir = repo_root / "accounting" / "notebooks" / "accounting_reports"
if str(reports_dir) not in sys.path:
    sys.path.insert(0, str(reports_dir))

from _shared import *

repo_root = find_repo_root(repo_root)
pack_dir = professional_pack_dir(repo_root)

artifact_inventory = inspect_artifacts(repo_root)
metrics = load_annual_dashboard_metrics(repo_root)
annual_qa = load_annual_dashboard_qa(repo_root)
debt_status = load_debt_status_reconciliation(repo_root)

years = available_years(metrics)
currencies = available_currencies(metrics)
extended_qa = extended_qa_findings(metrics, artifact_inventory, debt_status)

print("repo_root:", repo_root)
print("pack_dir:", pack_dir)
print("years:", years)
print("currencies:", currencies)
print("metric rows:", len(metrics))


repo_root: /home/matias/repos/accounting-backend
pack_dir: /home/matias/repos/accounting-backend/out/professional_pack/latest
years: ['2022', '2023', '2024', '2025', '2026']
currencies: ['ARS', 'N/A', 'USD']
metric rows: 350


## 1. QA extendido de lectura

Este bloque reconcilia la notebook anterior con los checks extendidos: métricas `available` sin valor, colisiones de etiquetas al ocultar `metric_id`, cash no frontend-safe, cross-currency y deuda engine-vs-ledger.


In [ ]:
display(short_status_table(extended_qa))
export_table(extended_qa, repo_root, "overview_extended_qa.csv")


,severity,area,check,n,detail
0,fail,artifacts,required_artifacts_exist,1,Missing: public_manifest
3,ok,cash,cash_frontend_safe_available,5,5 available cash rows
2,ok,currency,no_cross_currency_totals,0,No suspicious cross-currency Currency labels f...
1,ok,metrics,available_metric_has_value,0,No available metrics with NaN values
4,warning,display,hidden_metric_id_collisions,59,59 display key groups map to multiple metric_i...
5,warning,metrics,unavailable_visible,7,7 unavailable/blocked rows should be surfaced ...


wrote: /home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_extended_qa.csv rows=6 cols=5


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_extended_qa.csv')

## 2. Executive summary

Lectura profesional preliminar:

- La operación patrimonial parece generar resultado operativo positivo.
- Los retiros/distribuciones tienden a absorber gran parte del resultado operativo.
- La caja real todavía no es auditable si no existe `validated_cash_close`.
- La deuda interna en USD está mejor modelada y debería mirarse como stock/flow separado.
- El sistema opera como una red de gobernanza, pagos directos, claims y deuda; no como una caja única.


In [3]:
overview_specs = [
    spec("Overview", "Resultado operativo", "Renta total", "IS.RENT.TOTAL", 100,
         professional_comment="Ingreso operativo propio de la explotación patrimonial. No incluye funding ni deuda."),
    spec("Overview", "Resultado operativo", "OPEX propiedad", "IS.OPEX.PROPERTY", 200,
         professional_comment="Costos operativos de propiedad: impuestos, servicios, mantenimiento, legal y otros OPEX reales."),
    spec("Overview", "Resultado operativo", "Resultado operativo neto", "IS.NET.OPERATING", 300,
         professional_comment="Renta menos OPEX de propiedad. Es la medida más limpia de performance operativa."),
    spec("Overview", "Funding y distribuciones", "Funding / aportes", "FUND.CONTRIB.TOTAL", 400,
         professional_comment="Aportes para sostener caja o cubrir necesidades. No es ingreso operativo."),
    spec("Overview", "Funding y distribuciones", "Retiros / gasto personal", "DIST.DRAWS.PERSONAL", 500,
         professional_comment="Salidas hacia gasto personal/familiar. No es OPEX de propiedades."),
    spec("Overview", "Funding y distribuciones", "Dividendos", "DIST.DIVIDENDS", 510,
         professional_comment="Distribuciones patrimoniales. Deben separarse de gasto operativo."),
    spec("Overview", "Funding y distribuciones", "Cobertura después de funding y retiros", "COV.NET.AFTER_DRAWS", 600,
         professional_comment="Puente gerencial después de aportes y retiros. No debe llamarse profit contable."),
    spec("Overview", "Posición operativa", "Caja validada total", "BS.CASH.TOTAL", 700,
         professional_comment="Caja solo si existe fuente frontend-safe. Si no, corresponde s/d y no cero."),
    spec("Overview", "Posición operativa", "Deuda total abierta", "ID.DEBT.TOTAL.OPEN", 800,
         professional_comment="Stock de deuda interna abierta por moneda. No es gasto operativo."),
    spec("Overview", "Posición operativa", "Principal abierto", "ID.DEBT.PRINCIPAL.OPEN", 810,
         professional_comment="Componente principal de deuda abierta."),
    spec("Overview", "Posición operativa", "Interés abierto", "ID.DEBT.INTEREST.OPEN", 820,
         professional_comment="Componente interés/costo de oportunidad si el engine lo produce."),
]

overview_table = build_statement_table(metrics, overview_specs, years, include_debug_cols=False, drop_all_empty_years=False)

# Presentation-layer ratios for ARS where relevant.
overview_table = add_ratio_row(
    overview_table,
    section="Resultado operativo",
    line="Margen operativo",
    numerator_line_contains="Resultado operativo neto",
    denominator_line_contains="Renta total",
    years=years,
    currency="ARS",
    professional_comment="Resultado operativo neto / renta total. Ratio de presentación."
)
overview_table = add_ratio_row(
    overview_table,
    section="Resultado operativo",
    line="OPEX / renta",
    numerator_line_contains="OPEX propiedad",
    denominator_line_contains="Renta total",
    years=years,
    currency="ARS",
    professional_comment="OPEX propiedad / renta total. Ratio de presión operativa."
)
overview_table = add_ratio_row(
    overview_table,
    section="Funding y distribuciones",
    line="Retiros / resultado operativo",
    numerator_line_contains="Retiros / gasto personal",
    denominator_line_contains="Resultado operativo neto",
    years=years,
    currency="ARS",
    professional_comment="Retiros personales sobre resultado operativo. Si supera 100%, la operación no cubre retiros."
)

displayed_overview = display_statement(
    overview_table,
    "Overview anual",
    "Resumen ejecutivo de operación, funding/distribuciones, posición y deuda. Monedas integradas por fila, sin totales cross-currency.",
)
export_table(overview_table, repo_root, "overview_balance_dashboard.csv")


## Overview anual

Resumen ejecutivo de operación, funding/distribuciones, posición y deuda. Monedas integradas por fila, sin totales cross-currency.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Resultado operativo,Renta total,ARS,4.948.804,8.961.302,13.558.336,40.125.422,29.589.000,available,Ingreso operativo propio de la explotación patrimonial. No incluye funding ni deuda.,
Resultado operativo,Renta total,USD,s/d,s/d,3.610,4.560,2.280,available,Ingreso operativo propio de la explotación patrimonial. No incluye funding ni deuda.,
Resultado operativo,OPEX propiedad,ARS,587.266,4.158.761,8.066.302,12.168.481,7.422.915,available,"Costos operativos de propiedad: impuestos, servicios, mantenimiento, legal y otros OPEX reales.",
Resultado operativo,OPEX propiedad,USD,s/d,0,0,0,0,available,"Costos operativos de propiedad: impuestos, servicios, mantenimiento, legal y otros OPEX reales.",
Resultado operativo,Resultado operativo neto,ARS,4.361.538,4.802.541,5.492.034,27.956.941,22.166.085,available,Renta menos OPEX de propiedad. Es la medida más limpia de performance operativa.,
Resultado operativo,Resultado operativo neto,USD,s/d,0,3.610,4.560,2.280,available,Renta menos OPEX de propiedad. Es la medida más limpia de performance operativa.,
Funding y distribuciones,Funding / aportes,ARS,0,2.006.220,3.312.000,3.692.001,110.000,available,Aportes para sostener caja o cubrir necesidades. No es ingreso operativo.,
Funding y distribuciones,Funding / aportes,USD,s/d,0,0,0,0,available,Aportes para sostener caja o cubrir necesidades. No es ingreso operativo.,
Funding y distribuciones,Retiros / gasto personal,ARS,4.361.538,8.588.546,13.558.336,34.485.689,26.516.010,available,Salidas hacia gasto personal/familiar. No es OPEX de propiedades.,
Funding y distribuciones,Retiros / gasto personal,USD,s/d,0,100,0,380,available,Salidas hacia gasto personal/familiar. No es OPEX de propiedades.,


wrote: /home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_balance_dashboard.csv rows=21 cols=12


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_balance_dashboard.csv')

In [ ]:
def bridge_reconciliation(metrics, years, currency="ARS"):
    def series(metric_id):
        x = metrics[
            metrics["metric_id"].astype(str).eq(metric_id)
            & metrics["Currency"].astype(str).eq(currency)
        ].copy()
        return (
            x.groupby("period")["value"]
            .sum()
            .reindex(years)
            .astype(float)
        )

    operating = series("IS.NET.OPERATING")
    funding = series("FUND.CONTRIB.TOTAL")
    draws = series("DIST.DRAWS.PERSONAL")
    coverage = series("COV.NET.AFTER_DRAWS")


    residual = operating + funding - draws - coverage

    out = pd.DataFrame({
        "line": [
            "Resultado operativo",
            "+ Funding / aportes",
            "- Retiros y distribuciones totales",
            "= Cobertura después de funding y retiros",
            "Residual de reconciliación",
        ],
        "Currency": currency,
        **{
            y: [
                operating.get(y),
                funding.get(y),
                -draws.get(y),
                coverage.get(y),
                residual.get(y),
            ]
            for y in years
        }
    })

    return out


bridge_ars = bridge_reconciliation(metrics, years, "ARS")
display_statement(
    bridge_ars,
    "Reconciliación del puente de fondos — ARS",
    "Chequea que resultado + funding - retiros = cobertura. El residual debería ser cero o casi cero."
)
export_table(bridge_ars, repo_root, "overview_funds_bridge_reconciliation_ars.csv")

## Reconciliación del puente de fondos — ARS

Chequea que resultado + funding - retiros = cobertura. El residual debería ser cero o casi cero.

line,Currency,2022,2023,2024,2025,2026
Resultado operativo,ARS,4.361.538,4.802.541,5.492.034,27.956.941,22.166.085
+ Funding / aportes,ARS,0,2.006.220,3.312.000,3.692.001,110.000
- Retiros y distribuciones totales,ARS,-4.361.538,-8.588.546,-13.558.336,-34.485.689,-26.516.010
= Cobertura después de funding y retiros,ARS,0,-1.779.785,-4.754.302,-2.836.747,-4.239.925
Residual de reconciliación,ARS,0,0,0,0,-0


wrote: /home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_funds_bridge_reconciliation_ars.csv rows=5 cols=7


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_funds_bridge_reconciliation_ars.csv')

In [ ]:
def load_monthly_semantic(repo_root):
    path = repo_root / "out" / "run" / "accounting" / "latest" / "monthly_flow_semantic_split.csv"
    if not path.exists():
        print("Missing:", path)
        return pd.DataFrame()
    return pd.read_csv(path)


flow = load_monthly_semantic(repo_root)
print(flow.shape)
display(pd.DataFrame({"columns": flow.columns}))

(781, 25)


,columns
0,period
1,period_end
2,Currency
3,Box
4,Lugar
5,actor
6,counterparty
7,payer
8,receiver
9,channel


In [ ]:
def normalize_period_year(df):
    x = df.copy()
    if "period" in x.columns:
        x["year"] = x["period"].astype(str).str.slice(0, 4)
    elif "month" in x.columns:
        x["year"] = x["month"].astype(str).str.slice(0, 4)
    elif "Fecha" in x.columns:
        x["year"] = pd.to_datetime(x["Fecha"], errors="coerce").dt.year.astype("Int64").astype(str)
    else:
        x["year"] = ""
    return x


flow2 = normalize_period_year(flow)

value_col = "value" if "value" in flow2.columns else None
if value_col is None:
    for c in ["amount", "Amount", "monto", "Monto", "signed_amount"]:
        if c in flow2.columns:
            value_col = c
            break

# if value_col is None:
#     raise ValueError("Could not find value column in monthly_flow_semantic_split")

# flow2[value_col] = pd.to_numeric(flow2[value_col], errors="coerce")

cols_to_show = [
    c for c in [
        "year",
        "period",
        "Currency",
        "Box",
        "box",
        "governance_box",
        "payer",
        "receiver",
        "actor",
        "counterparty",
        "semantic_bucket",
        "semantic_subbucket",
        "cash_path",
        "flow_type",
        "Tipo",
        "source_table",
        value_col,
    ]
    if c in flow2.columns
]

f2024 = flow2[
    (flow2["year"] == "2024").astype(bool)
    & flow2.get("Currency", pd.Series("", index=flow2.index)).astype(str).eq("ARS")
].copy()

display(f2024[cols_to_show].head(100))

,year,period,Currency,Box,payer,receiver,actor,counterparty,semantic_bucket,semantic_subbucket,cash_path,source_table
221,2024,2024-01,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
222,2024,2024-01,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
223,2024,2024-01,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
224,2024,2024-01,ARS,Family Business,FB,Gasto,Family Business,Gasto,family_withdrawal_candidate,personal_expense,Transfer:Gasto,ledger_canonical.csv
225,2024,2024-01,ARS,Household,MI,HH,Household,MI,funding_contribution,family_or_tenant_contribution,Contribucion:Contribuciones,ledger_canonical.csv
226,2024,2024-01,ARS,Household,HH,Servicios,Household,Servicios,property_opex,services,Pagos:Servicio,ledger_canonical.csv
227,2024,2024-01,ARS,Property Management,PM,Costos,Property Management,Costos,property_opex,legal,Pagos:Legal,ledger_canonical.csv
228,2024,2024-01,ARS,Property Management,MI,PM,Property Management,MI,property_opex,legal,Contribucion:Legal,ledger_canonical.csv
229,2024,2024-02,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
230,2024,2024-02,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv


In [ ]:
f2024.sample(20)

,period,period_end,Currency,Box,Lugar,actor,counterparty,payer,receiver,channel,cash_path,semantic_bucket,semantic_subbucket,amount_in,amount_out,net_amount,amount_abs,n_tx,classification_status,classification_confidence,review_required,source_table,source_tx_ids_sample,rule_ids,notes,year
367,2024-10,2024-10-31,ARS,Property Management,CABA,Property Management,Impuestos,PM,Impuestos,Property Management,Pagos:Impuestos,property_opex,taxes,0.0,119602.0,-119602.0,119602.0,1,classified,high,False,ledger_canonical.csv,388309d9d9f6d003,R002_property_taxes,NaN,2024
286,2024-06,2024-06-30,ARS,Household,CABA,Household,MI,MI,HH,Household,Contribucion:Contribuciones,funding_contribution,family_or_tenant_contribution,140000.0,0.0,140000.0,140000.0,1,classified,high,False,ledger_canonical.csv,881929a06cac7a64,R006_contribution,NaN,2024
420,2024-12,2024-12-31,ARS,Property Management,CABA,Property Management,MI,MI,PM,Property Management,Contribucion:Servicio,property_opex,services,452484.0,0.0,452484.0,452484.0,1,classified,high,False,ledger_canonical.csv,d87c53eb189ca5d9,R003_property_services,NaN,2024
260,2024-04,2024-04-30,ARS,Property Management,NaN,Property Management,MI,PM,MI,Property Management,Transfer:Repago,debt_movement,repayment,0.0,53243.0,-53243.0,53243.0,1,classified,high,False,ledger_canonical.csv,c72ff53df4c5f24b,R008_debt_repayment,NaN,2024
224,2024-01,2024-01-31,ARS,Family Business,NaN,Family Business,Gasto,FB,Gasto,Family Business,Transfer:Gasto,family_withdrawal_candidate,personal_expense,0.0,765890.0,-765890.0,765890.0,1,classified,medium,False,ledger_canonical.csv,a6fd814ecb2d2843,R011_personal_expense_text,review family/informal withdrawal candidate,2024
336,2024-09,2024-09-30,ARS,Family Business,Tigre 01,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,516127.0,0.0,516127.0,516127.0,3,classified,high,False,ledger_canonical.csv,a38d5fdbb0bf95dd;a2bbe396e76b45f6;5bdc888afdd4...,R001_rent_collections,NaN,2024
359,2024-10,2024-10-31,ARS,Household,CABA,Household,Alen,Alen,HH,Household,Contribucion:Contribuciones,funding_contribution,family_or_tenant_contribution,150000.0,0.0,150000.0,150000.0,1,classified,high,False,ledger_canonical.csv,4ffbbea35a9c617a,R006_contribution,NaN,2024
265,2024-05,2024-05-31,ARS,Family Business,Tigre 32,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,342556.0,0.0,342556.0,342556.0,3,classified,high,False,ledger_canonical.csv,5609904d55a476d3;c1633aa18d6aacda;b03053f0715f...,R001_rent_collections,NaN,2024
327,2024-08,2024-08-31,ARS,Household,CABA,Household,Cande,Cande,HH,Household,Contribucion:Servicio,property_opex,services,35500.0,0.0,35500.0,35500.0,1,classified,high,False,ledger_canonical.csv,d944b9acfebdabd1,R003_property_services,NaN,2024
423,2024-12,2024-12-31,ARS,Property Management,Tigre 01,Property Management,Impuestos,PM,Impuestos,Property Management,Pagos:Impuestos,property_opex,taxes,0.0,147638.0,-147638.0,147638.0,1,classified,high,False,ledger_canonical.csv,2a67de1891c84ee6,R002_property_taxes,NaN,2024


In [8]:
fb_cols = [c for c in ["Box", "box", "governance_box", "payer", "receiver", "actor", "counterparty"] if c in f2024.columns]

mask_fb = pd.Series(False, index=f2024.index)
for c in fb_cols:
    mask_fb = mask_fb | f2024[c].astype(str).str.contains("Family Business|FB", case=False, na=False)

fb2024 = f2024[mask_fb].copy()

print("FB-related 2024 rows:", len(fb2024))
display(fb2024[cols_to_show].sort_values([c for c in ["period", "semantic_bucket", "semantic_subbucket"] if c in fb2024.columns]).head(200))

FB-related 2024 rows: 51


,year,period,Currency,Box,payer,receiver,actor,counterparty,semantic_bucket,semantic_subbucket,cash_path,source_table
224,2024,2024-01,ARS,Family Business,FB,Gasto,Family Business,Gasto,family_withdrawal_candidate,personal_expense,Transfer:Gasto,ledger_canonical.csv
221,2024,2024-01,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
222,2024,2024-01,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
223,2024,2024-01,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
232,2024,2024-02,ARS,Family Business,FB,Gasto,Family Business,Gasto,family_withdrawal_candidate,personal_expense,Transfer:Gasto,ledger_canonical.csv
229,2024,2024-02,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
230,2024,2024-02,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
231,2024,2024-02,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv
239,2024,2024-03,ARS,Family Business,FB,Gasto,Family Business,Gasto,family_withdrawal_candidate,personal_expense,Transfer:Gasto,ledger_canonical.csv
236,2024,2024-03,ARS,Family Business,Inq,FB,Family Business,Inq,operating_revenue,rent,Cobros:Renta,ledger_canonical.csv


In [ ]:
fb2024.sort_values([c for c in ["period", "semantic_bucket", "semantic_subbucket"] if c in fb2024.columns])

,period,period_end,Currency,Box,Lugar,actor,counterparty,payer,receiver,channel,cash_path,semantic_bucket,semantic_subbucket,amount_in,amount_out,net_amount,amount_abs,n_tx,classification_status,classification_confidence,review_required,source_table,source_tx_ids_sample,rule_ids,notes,year
224,2024-01,2024-01-31,ARS,Family Business,NaN,Family Business,Gasto,FB,Gasto,Family Business,Transfer:Gasto,family_withdrawal_candidate,personal_expense,0.0,765890.0,-765890.0,765890.0,1,classified,medium,False,ledger_canonical.csv,a6fd814ecb2d2843,R011_personal_expense_text,review family/informal withdrawal candidate,2024
221,2024-01,2024-01-31,ARS,Family Business,Tigre 01,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,367823.0,0.0,367823.0,367823.0,3,classified,high,False,ledger_canonical.csv,254f6f0a8633a959;d47323313841de72;73681a97dc10...,R001_rent_collections,NaN,2024
222,2024-01,2024-01-31,ARS,Family Business,Tigre 28,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,229322.0,0.0,229322.0,229322.0,2,classified,high,False,ledger_canonical.csv,6b9525b7f4c457b5;ebda5d099d229953,R001_rent_collections,NaN,2024
223,2024-01,2024-01-31,ARS,Family Business,Tigre 32,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,168745.0,0.0,168745.0,168745.0,2,classified,high,False,ledger_canonical.csv,4953edf8f1d97f1a;25d8294653a4cd19,R001_rent_collections,NaN,2024
232,2024-02,2024-02-29,ARS,Family Business,NaN,Family Business,Gasto,FB,Gasto,Family Business,Transfer:Gasto,family_withdrawal_candidate,personal_expense,0.0,883490.0,-883490.0,883490.0,1,classified,medium,False,ledger_canonical.csv,e735684fbe3ed901,R011_personal_expense_text,review family/informal withdrawal candidate,2024
229,2024-02,2024-02-29,ARS,Family Business,Tigre 01,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,385423.0,0.0,385423.0,385423.0,3,classified,high,False,ledger_canonical.csv,be28e81ce6388076;f4941495cf33cd95;8f4623d48142...,R001_rent_collections,NaN,2024
230,2024-02,2024-02-29,ARS,Family Business,Tigre 28,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,229322.0,0.0,229322.0,229322.0,2,classified,high,False,ledger_canonical.csv,920af0a7f85e89d4;88d1c989bcff5d05,R001_rent_collections,NaN,2024
231,2024-02,2024-02-29,ARS,Family Business,Tigre 32,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,268745.0,0.0,268745.0,268745.0,3,classified,high,False,ledger_canonical.csv,960425489a0a731f;ba48284b16fb0974;182b0e0439c1...,R001_rent_collections,NaN,2024
239,2024-03,2024-03-31,ARS,Family Business,NaN,Family Business,Gasto,FB,Gasto,Family Business,Transfer:Gasto,family_withdrawal_candidate,personal_expense,0.0,883490.0,-883490.0,883490.0,1,classified,medium,False,ledger_canonical.csv,226b8b61bca047a6,R011_personal_expense_text,review family/informal withdrawal candidate,2024
236,2024-03,2024-03-31,ARS,Family Business,Tigre 01,Family Business,Inq,Inq,FB,Family Business,Cobros:Renta,operating_revenue,rent,385423.0,0.0,385423.0,385423.0,3,classified,high,False,ledger_canonical.csv,e1e5ce87a85b1fce;5ec1016e4d42841f;185747f34db9...,R001_rent_collections,NaN,2024


In [10]:
# group_cols = [
#     c for c in [
#         "semantic_bucket",
#         "semantic_subbucket",
#         "cash_path",
#         "Box",
#         "box",
#         "governance_box",
#         "payer",
#         "receiver",
#         "actor",
#         "counterparty",
#     ]
#     if c in fb2024.columns
# ]

# fb_summary = (
#     fb2024
#     .groupby(group_cols, dropna=False)[value_col]
#     .sum()
#     .reset_index()
#     .sort_values(value_col, ascending=False)
# )

# display(fb_summary.head(100))
# export_table(fb_summary, repo_root, "debug_2024_fb_drain_trace.csv")

In [11]:
draws_by_type = metrics[
    metrics["metric_id"].astype(str).isin([
        "DIST.DRAWS.PERSONAL",
        "DIST.DIVIDENDS",
        "DIST.DRAWS.BY_TYPE",
        "DQ.UNKNOWN.AMOUNT",
    ])
].copy()

display(
    draws_by_type[
        [
            c for c in [
                "metric_id",
                "dashboard_section",
                "Currency",
                "period",
                "dimension_name",
                "dimension_value",
                "value",
                "value_status",
                "source_table",
                "caveat",
            ]
            if c in draws_by_type.columns
        ]
    ].sort_values(["metric_id", "Currency", "period", "dimension_name", "dimension_value"])
)

,metric_id,dashboard_section,Currency,period,dimension_name,dimension_value,value,value_status,source_table,caveat
45,DIST.DIVIDENDS,2. Funding and distributions,ARS,2022,,,0.0,available,monthly_operating_statement.csv,
46,DIST.DIVIDENDS,2. Funding and distributions,ARS,2023,,,0.0,available,monthly_operating_statement.csv,
48,DIST.DIVIDENDS,2. Funding and distributions,ARS,2024,,,0.0,available,monthly_operating_statement.csv,
50,DIST.DIVIDENDS,2. Funding and distributions,ARS,2025,,,0.0,available,monthly_operating_statement.csv,
52,DIST.DIVIDENDS,2. Funding and distributions,ARS,2026,,,1367384.0,available,monthly_operating_statement.csv,
47,DIST.DIVIDENDS,2. Funding and distributions,USD,2023,,,0.0,available,monthly_operating_statement.csv,
49,DIST.DIVIDENDS,2. Funding and distributions,USD,2024,,,100.0,available,monthly_operating_statement.csv,
51,DIST.DIVIDENDS,2. Funding and distributions,USD,2025,,,0.0,available,monthly_operating_statement.csv,
53,DIST.DIVIDENDS,2. Funding and distributions,USD,2026,,,380.0,available,monthly_operating_statement.csv,
183,DIST.DRAWS.BY_TYPE,2. Funding and distributions,ARS,2022,semantic_subbucket,personal_expense,4361538.0,available,monthly_flow_semantic_split.csv,


## 3. Comentarios profesionales de overview

Este bloque no busca cerrar diagnóstico legal/contable, sino dejar explícitas las tensiones que deben discutirse con contadores y gente de negocio.


In [12]:
overview_comments = pd.DataFrame([
    {"tema": "Operación", "comentario": "El resultado operativo debe aislar rentas y OPEX reales de propiedad. Funding, deuda, dividendos y gasto personal van fuera."},
    {"tema": "Distribución", "comentario": "Si retiros/dividendos absorben más que el resultado operativo, el problema es de política de distribución y gobernanza, no solo de rentabilidad."},
    {"tema": "Caja", "comentario": "Caja validada debe permanecer s/d si no hay validated_cash_close frontend-safe. No se debe usar daily_cash_position como caja reportable."},
    {"tema": "Deuda", "comentario": "La deuda interna muestra quién financió o adelantó fondos. Debe tratarse como stock/flow separado y reconciliado."},
    {"tema": "Entidad", "comentario": "Hace falta decidir perspectiva: PM, familia consolidada o sistema patrimonial total. Activo/pasivo depende de esa entidad."},
    {"tema": "2026", "comentario": "Si 2026 es año parcial, todos los cuadros deben marcar YTD para evitar comparaciones engañosas."},
])
display(overview_comments)
export_table(overview_comments, repo_root, "overview_professional_comments.csv")


,tema,comentario
0,Operación,El resultado operativo debe aislar rentas y OP...
1,Distribución,Si retiros/dividendos absorben más que el resu...
2,Caja,Caja validada debe permanecer s/d si no hay va...
3,Deuda,La deuda interna muestra quién financió o adel...
4,Entidad,"Hace falta decidir perspectiva: PM, familia co..."
5,2026,"Si 2026 es año parcial, todos los cuadros debe..."


wrote: /home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_professional_comments.csv rows=6 cols=2


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/overview_professional_comments.csv')

## 4. Export report

Exporta HTML y Markdown simples a `out/professional_pack/latest/`.


In [13]:
md_path = write_markdown_report(
    repo_root,
    "01_balance_dashboard_overview.md",
    "01 — Balance dashboard overview",
    [
        ("Executive summary", "La operación, funding/distribuciones, caja y deuda se presentan como capas separadas. ARS y USD conviven por fila, sin sumarse."),
        ("Overview table", overview_table),
        ("Extended QA", extended_qa),
        ("Professional comments", overview_comments),
    ],
)
html_path = write_html_report(
    repo_root,
    "01_balance_dashboard_overview.html",
    "01 — Balance dashboard overview",
    [
        ("Executive summary", "La operación patrimonial parece positiva, pero la distribución/retiros, caja no auditada y deuda interna requieren tratamiento separado."),
        ("Overview table", overview_table),
        ("Extended QA", extended_qa),
        ("Professional comments", overview_comments),
    ],
)
print("markdown:", md_path.relative_to(repo_root))
print("html:", html_path.relative_to(repo_root))


markdown: out/professional_pack/latest/markdown/01_balance_dashboard_overview.md
html: out/professional_pack/latest/html/01_balance_dashboard_overview.html
